# 04 — The protein layer

**Does the transcriptional story hold at protein level?**

The four quadrants of an RNA-vs-protein log2FC comparison:

| | protein down | protein flat |
|---|---|---|
| **RNA down** | concordant loss of function | buffering, protein half-life, or ADT floor |
| **RNA flat** | **post-transcriptional regulation** | no effect |

The bottom-left quadrant is the mechanistically interesting one, and it is
invisible to any RNA-only screen.

**Positive control:** the published mechanism is that CD58 protein is *not*
IFN-γ-inducible while MHC-I is, and that CD58 loss confers immune evasion
without compromising MHC. Recovering that independently validates the whole
pipeline.

In [ ]:
# =============================================================================
# nb04 — ADT: quality control and cross-modality structure
#
# Two questions:
#   1. Which of the 20 surface targets are measuring signal rather than
#      background? The panel carries 4 matched isotype controls, so this is
#      answerable directly rather than by eye.
#   2. Does protein-defined structure agree with RNA-defined structure? The
#      RNA embedding from nb03 is the reference; ADT clusters are computed
#      independently and overlaid.
#
# Normalisation note: CLR is per cell across features, so it divides out total
# staining intensity by construction. Any "overall ADT level" score computed
# after CLR is therefore near-zero for every cell and carries nothing. Total
# capture is tracked separately as adt.obs["ncounts"], as QC.
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import muon as mu
from adjustText import adjust_text

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)
sc.settings.verbosity = 1

s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]
COND       = s["condition"]
GUIDE      = s["guide"]
CTRL       = cfg["schema"]["control_label"]
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]

# isotypes carry NaN in the Isotype_control column; targets name their match
isotypes = adt.var_names[adt.var["Isotype_control"].isna()].tolist()
targets  = adt.var_names[adt.var["Isotype_control"].notna()].tolist()
iso_map  = adt.var.loc[targets, "Isotype_control"].to_dict()

print(f"ADT: {adt.n_obs:,} cells x {adt.n_vars} features")
print(f"targets:  {len(targets)}")
print(f"isotypes: {isotypes}")
print(f"\nADT counts per cell: median {np.median(adt.obs['ncounts']):.0f}, "
      f"range {adt.obs['ncounts'].min():.0f}-{adt.obs['ncounts'].max():.0f}")

In [ ]:
# load UMAP embeddings from RNA GEX from last notebook, so you can overlay ADT signal in this notebook.

emb = pd.read_parquet(P.data_interim / "03_embedding.parquet")
emb = emb.loc[adt.obs_names]          # align to ADT cell order
print(f"embedding: {len(emb):,} cells, clusters: {emb['cluster'].nunique()}")

In [ ]:
# ---- normalise and assemble a tidy CLR matrix ------------------------------
adt_n = adt.copy()
adt_n.layers["counts"] = adt_n.X.copy()
mu.prot.pp.clr(adt_n, axis=cfg["protein"]["clr_margin"])   # per cell, across features

X = adt_n.X
X = np.asarray(X.todense()) if sp.issparse(X) else np.asarray(X)
clr = pd.DataFrame(X, index=adt_n.obs_names, columns=adt_n.var_names)

raw = adt.layers["counts"] if "counts" in adt.layers else adt.X
raw = np.asarray(raw.todense()) if sp.issparse(raw) else np.asarray(raw)
raw = pd.DataFrame(raw, index=adt.obs_names, columns=adt.var_names)

# ---- per-feature signal statistics ----------------------------------------
# Two independent things determine whether a feature is usable:
#   separation  — how far above its OWN matched isotype does it sit?
#   dynamic range — does it vary across cells, or sit flat at one level?
# A feature can clear one and fail the other. Flat-but-high is uninformative
# for finding perturbation effects even though it is "real" staining.
rows = []
for t in targets:
    iso = iso_map[t]
    if iso not in clr.columns:
        continue
    v = clr[t]
    rows.append({
        "feature": t,
        "isotype": iso,
        "mean_clr": v.mean(),
        "iso_mean_clr": clr[iso].mean(),
        # mean gap above matched background
        "separation": v.mean() - clr[iso].mean(),
        # gap at the top of the distribution — finds a positive subpopulation
        # that a mean or median would miss for an on/off marker
        "p90_separation": v.quantile(0.9) - clr[iso].quantile(0.9),
        # spread across cells: 90th - 10th percentile
        "dynamic_range": v.quantile(0.9) - v.quantile(0.1),
        "median_counts": raw[t].median(),
        "pct_positive": (raw[t] > 0).mean() * 100,
    })
qc = pd.DataFrame(rows).set_index("feature")

# thresholds are conventions, not standards — stated so they can be argued with
SEP_MIN, DR_MIN = 0.5, 0.5
qc["usable"] = (qc["separation"] >= SEP_MIN) & (qc["dynamic_range"] >= DR_MIN)
qc = qc.sort_values("separation", ascending=False)

print(qc.round(2).to_string())
print(f"\nusable: {qc['usable'].sum()} / {len(qc)}")
print(f"failing: {qc.index[~qc['usable']].tolist()}")
qc.to_csv(P.tables / "04_adt_feature_qc.csv")
usable_features = qc.index[qc["usable"]].tolist()

In [ ]:
(raw["HLA_E"] > 0).groupby(adt.obs[COND].values).mean()


In [ ]:
struct = []
for t in targets:
    iso = iso_map.get(t)
    if iso is None or str(iso) == "nan" or iso not in clr.columns:
        continue
    pos = raw[t] > 0
    v = clr[t]
    by_cond  = v.groupby(adt.obs[COND].values).mean()
    by_clust = v.groupby(emb["cluster"].values).mean()
    struct.append({
        "feature": t,
        "pct_positive": pos.mean() * 100,
        # signal where detected, vs matched background
        "clr_pos_vs_iso": clr.loc[pos, t].mean() - clr[iso].mean(),
        # structure in LEVEL, which has room to vary even at 100% detection
        "cond_range_clr":  by_cond.max()  - by_cond.min(),
        "clust_range_clr": by_clust.max() - by_clust.min(),
        # detection-rate structure, informative only for on/off features
        "clust_range_det": pos.groupby(emb["cluster"].values).mean().pipe(
            lambda s: s.max() - s.min()),
    })
struct = pd.DataFrame(struct).set_index("feature")
print(struct.sort_values("clust_range_clr", ascending=False).round(3).to_string())

In [ ]:
# the above was attempt to filter for "good" ADT signal to use in clustering.
# But the logic didn't make much sense - looking for ADT signal that is absent in most cells
# rationale being that this will drive clustering too much, and spuriously
# but this could be real - small subsets that express this protein
# lets first look at the ADT expr profiles

# =============================== FIGURE ====================================
# ADT CLR distributions, one panel per isotype family. All cells pooled.
#
# Splitting by isotype family puts each target next to ITS OWN background —
# the only comparison that means anything, since non-specific binding differs
# by host species and subclass. It also frees the full colour range within
# each panel, which 20 overlaid curves on one axis could not support.
#
# KDE curve over a light stepped histogram of the same data. The histogram is
# kept because ADT counts are integers, so CLR is heavily discretised at the
# low end — the KDE smooths over that, and showing the bins keeps the
# underlying granularity visible rather than hiding it.
import matplotlib.colors as mcolors
from scipy.stats import gaussian_kde

families = ["Mouse_IgG1", "Mouse_IgG2a", "Mouse_IgG2b", "Rat_IgG2a"]
families = [f for f in families if f in clr.columns]

lo = clr.values.min()
hi = np.percentile(clr[targets].values, 99.9)
bins = np.linspace(lo, hi, 40)            # coarser than before; KDE carries shape
xs   = np.linspace(lo, hi, 400)
BW   = 0.15                               # KDE bandwidth; lower = less smoothing
Y_MAX = 50_000

rng = np.random.default_rng(SEED)
sub = rng.choice(len(clr), min(20_000, len(clr)), replace=False)   # KDE is slow


def lighten(c, f=0.80):
    r, g, b = mcolors.to_rgb(c)
    return (r + (1 - r) * f, g + (1 - g) * f, b + (1 - b) * f)


def kde_counts(v):
    """KDE scaled from density back to cell counts, so it overlays the hist."""
    k = gaussian_kde(v[sub], bw_method=BW)
    return k(xs) * len(v) * (bins[1] - bins[0])


fig, axes = plt.subplots(len(families), 1, figsize=(13, 3.6 * len(families)),
                         sharex=True)
axes = np.atleast_1d(axes)

for a, iso in zip(axes, families):
    members = sorted([t for t in targets if iso_map.get(t) == iso])

    # ---- background: stepped hist + heavy KDE line ------------------------
    a.hist(clr[iso], bins=bins, histtype="stepfilled", color="#e6e6e6",
           edgecolor="#cccccc", lw=0.6, zorder=1)
    a.plot(xs, kde_counts(clr[iso].values), lw=3.0, color="#7d7d7d", zorder=2,
           label=f"{iso.replace('_',' ')} (isotype)")

    # ---- targets ----------------------------------------------------------
    cmap = plt.cm.turbo(np.linspace(0.05, 0.95, max(len(members), 2)))
    for i, t in enumerate(members):
        a.hist(clr[t], bins=bins, histtype="stepfilled",
               color=lighten(cmap[i]), alpha=0.30,
               edgecolor=lighten(cmap[i], 0.55), lw=0.6, zorder=3)
        a.plot(xs, kde_counts(clr[t].values), lw=2.2, color=cmap[i], zorder=4,
               label=f"{t} ({struct.loc[t,'pct_positive']:.0f}%)")

    a.set_ylim(0, Y_MAX)
    a.set_xlim(lo, hi)
    a.set_ylabel("number of cells")
    a.legend(fontsize=7, ncol=2, loc="upper right", framealpha=0.92)
    a.set_title(f"{iso.replace('_',' ')} family — {len(members)} targets",
                loc="center", fontsize=11)

axes[-1].set_xlabel("CLR")

for i, a in enumerate(axes):
    a.set_title(f"{chr(65+i)})", loc="left", fontweight="bold",
                fontsize=13, pad=8)

fig.suptitle("ADT signal vs matched isotype control",
             fontweight="bold", fontsize=14, y=0.999)
fig.tight_layout()
savefig(fig, "04_adt_distributions", cfg)





In [ ]:
# =============================================================================
# ADT-based clustering
#
# All 20 target antibodies retained. The isotype comparison showed most every
# feature sits above its matched background where it binds
# (clr_pos_vs_iso 0.44-0.66, tight range) and every feature varies across
# clusters — so nothing is dead. The panel splits into constitutive markers
# (near-100% positive, varying in level) and subset markers (26-58% positive,
# varying in who has them), and both carry information.
#
# Isotypes are excluded: they measure background by design, and including them
# would let non-specific binding contribute to inter-cell distances.
#
# Scaling matters here. After CLR the features have very different variances
# — CD49f spans ~1.0 in cluster means, CD202b ~0.29 — so without scaling the
# high-variance antibodies would dominate the distance metric. With no prior
# reason to weight one antibody over another, unit variance is the right
# default.

adt_c = adt_n[:, targets].copy()          # adt_n is already CLR-normalised
print(f"clustering on {adt_c.n_vars} targets, {adt_c.n_obs:,} cells")

sc.pp.scale(adt_c, max_value=10)

# 20 features, so ask for fewer components than the RNA side used
sc.tl.pca(adt_c, n_comps=15, svd_solver="arpack", random_state=SEED)
sc.pl.pca_variance_ratio(adt_c, n_pcs=15, log=True)

In [ ]:
N_PCS_ADT =8                            # set from the elbow above

sc.pp.neighbors(adt_c, n_neighbors=15, n_pcs=N_PCS_ADT, random_state=SEED)
sc.tl.leiden(adt_c, resolution=0.5, key_added="adt_cluster",
             random_state=SEED, flavor="igraph", n_iterations=2, directed=False)
sc.tl.umap(adt_c, random_state=SEED)      # ADT-only embedding, for comparison

print(adt_c.obs["adt_cluster"].value_counts().sort_index())

# ---- the question: does protein-defined structure match RNA-defined? -------
adt_c.obs["rna_cluster"] = emb.loc[adt_c.obs_names, "cluster"].values
adt_c.obs[COND] = adt.obs[COND].values

xtab = pd.crosstab(adt_c.obs["adt_cluster"], adt_c.obs["rna_cluster"],
                   normalize="index")
print("\nADT cluster x RNA cluster (row-normalised):")
print(xtab.round(2).to_string())

print("\nADT cluster x condition:")
print(pd.crosstab(adt_c.obs["adt_cluster"], adt_c.obs[COND],
                  normalize="index").round(2).to_string())

In [ ]:
# =============================== FIGURE ====================================
# Cross-modality structure. Does surface phenotype partition cells the same
# way the transcriptome does?
#
# Cluster numbering is arbitrary and does NOT correspond between modalities —
# ADT cluster 3 has no relation to RNA cluster 3. The crosstab is what maps
# them onto each other; these panels show the same question visually.
fig, ax = plt.subplot_mosaic(
    """
    AB
    """,
    figsize=(18, 7),
)

rna_emb = emb.loc[adt_c.obs_names, ["umap1", "umap2"]].values
adt_emb = adt_c.obsm["X_umap"]

# ---------------------------------- A --------------------------------------
# ADT-derived cluster labels on the RNA-derived embedding. Tight, coherent
# patches mean the two modalities agree; labels smeared across the map mean
# protein groups cells the transcriptome considers unrelated.
acl = adt_c.obs["adt_cluster"].astype(str)
aorder = sorted(acl.unique(), key=int)
acmap = plt.cm.tab20(np.linspace(0, 1, len(aorder)))

for i, cl in enumerate(aorder):
    m = (acl == cl).values
    ax["A"].scatter(rna_emb[m, 0], rna_emb[m, 1], s=1.2, alpha=0.5,
                    color=acmap[i], linewidths=0, rasterized=True,
                    label=f"{cl}  ({m.sum():,})")
    # median, not mean — robust to stragglers pulling the label off-cluster
    ax["A"].text(np.median(rna_emb[m, 0]), np.median(rna_emb[m, 1]), cl,
                 fontsize=9, fontweight="bold", ha="center", va="center",
                 bbox=dict(boxstyle="circle,pad=0.2", facecolor="white",
                           edgecolor="none", alpha=0.75))

ax["A"].set_xlabel("RNA UMAP 1"); ax["A"].set_ylabel("RNA UMAP 2")
ax["A"].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.01, 0.5),
               markerscale=8, framealpha=0.95, title="ADT cluster",
               title_fontsize=8)

# ---------------------------------- B --------------------------------------
# The mirror: RNA-derived labels on the ADT-derived embedding. Asymmetry
# between A and B is itself informative — one modality can resolve structure
# the other misses without the reverse holding.
rcl = adt_c.obs["rna_cluster"].astype(str)
rorder = sorted(rcl.unique(), key=int)
rcmap = plt.cm.tab20(np.linspace(0, 1, len(rorder)))

for i, cl in enumerate(rorder):
    m = (rcl == cl).values
    ax["B"].scatter(adt_emb[m, 0], adt_emb[m, 1], s=1.2, alpha=0.5,
                    color=rcmap[i], linewidths=0, rasterized=True,
                    label=f"{cl}  ({m.sum():,})")
    ax["B"].text(np.median(adt_emb[m, 0]), np.median(adt_emb[m, 1]), cl,
                 fontsize=9, fontweight="bold", ha="center", va="center",
                 bbox=dict(boxstyle="circle,pad=0.2", facecolor="white",
                           edgecolor="none", alpha=0.75))

ax["B"].set_xlabel("ADT UMAP 1"); ax["B"].set_ylabel("ADT UMAP 2")
ax["B"].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.01, 0.5),
               markerscale=8, framealpha=0.95, title="RNA cluster",
               title_fontsize=8)

for k in ["A", "B"]:
    ax[k].set_xticks([]); ax[k].set_yticks([])

titles = {
    "A": "ADT clusters on RNA embedding",
    "B": "RNA clusters on ADT embedding",
}
for label, a in ax.items():
    a.set_title(f"{label}) {titles.get(label, '')}", loc="left",
                fontweight="bold", fontsize=12, pad=8)

fig.tight_layout()
savefig(fig, "04_cross_modality", cfg)

In [ ]:
panels = load_panels()
print(panels["adt"].keys())

# =============================== FIGURE ====================================
# Every ADT feature by condition, CLR. Ordered by effect size, so the movers
# sort to the front and flat features to the back — which features DON'T
# respond is as informative as which do.
#
# Background tint marks direction of change vs Control, per condition band:
#   green = up      yellow = down      untinted = |Δ| below threshold
# Tinting the band rather than the whole panel lets a feature be up in one
# condition and down in the other without the encoding having to choose.
#
# CD58 is the built-in negative control: the published mechanism is that CD58
# protein is not IFN-γ-inducible while MHC-I is. If HLA_A rises and CD58 stays
# flat, that result is recovered independently here.
#
# Two features need reading with care. HLA_A is clone W6/32 — a conformational
# pan-MHC-I epitope requiring β2-microglobulin, so it reports assembled surface
# complex across HLA-A/B/C, not HLA-A alone. CD279 (PD-1) is a T-cell receptor,
# so its co-culture signal partly reflects the 577 contaminating lymphocytes
# identified in nb03 rather than melanoma biology.
from matplotlib.patches import Patch

adt_annot = panels["adt"]["annotations"]

clr_cond = clr.copy()
clr_cond[COND] = adt.obs[COND].values

means = clr_cond.groupby(COND, observed=True)[targets].mean().reindex(cond_order)
delta = means - means.loc["Control"]          # change vs baseline
print(delta.T.round(2).to_string())           # check TINT_MIN against this

TINT_MIN = 0.10                                # |Δ| below this is left untinted
UP, DOWN = "#d6f0d6", "#faf0c0"                # pale green, pale yellow

order_f = struct.sort_values("cond_range_clr", ascending=False).index.tolist()

ncol = 5
nrow = int(np.ceil(len(order_f) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.8 * nrow),
                         sharey=True)
axes = axes.ravel()

for a, t in zip(axes, order_f):
    # tint each condition band by its own direction, before the violins
    for j, cond in enumerate(cond_order):
        if cond == "Control":
            continue
        d = delta.loc[cond, t]
        if abs(d) >= TINT_MIN:
            a.axvspan(j - 0.5, j + 0.5, color=UP if d > 0 else DOWN, zorder=0)

    sns.violinplot(data=clr_cond, x=COND, y=t, order=cond_order,
                   hue=COND, hue_order=cond_order, palette=pal,
                   legend=False, cut=0, linewidth=0.7, ax=a, zorder=2)

    # medians as white points — violins hide them at this width
    med = clr_cond.groupby(COND, observed=True)[t].median().reindex(cond_order)
    a.scatter(range(len(cond_order)), med.values, color="white", s=16,
              zorder=10, edgecolor="black", linewidth=0.5)

    # matched isotype mean — distinguishes "signal fell" from "signal reached
    # background and is no longer detectable"
    iso = iso_map.get(t)
    if iso and str(iso) != "nan" and iso in clr.columns:
        a.axhline(clr[iso].mean(), ls="--", c="grey", lw=1, zorder=3)

    a.set_title(f"{t}\n{adt_annot.get(t, '')}\n"
                f"IFNγ {delta.loc['IFNγ', t]:+.2f}   "
                f"co-cult {delta.loc['Co-culture', t]:+.2f}",
                fontsize=8.5, linespacing=1.4)
    a.set_xlabel("")
    a.set_ylabel("CLR")
    a.set_xlim(-0.5, 2.5)
    a.tick_params(axis="x", rotation=25, labelsize=8)

for a in axes[len(order_f):]:
    a.set_axis_off()

fig.legend(handles=[
    Patch(facecolor=UP,   label=f"up vs Control (Δ ≥ {TINT_MIN})"),
    Patch(facecolor=DOWN, label=f"down vs Control (Δ ≤ −{TINT_MIN})"),
    plt.Line2D([], [], ls="--", c="grey", label="matched isotype mean"),
], loc="lower center", ncol=3, fontsize=9, frameon=False,
   bbox_to_anchor=(0.5, -0.005))

fig.suptitle("ADT signal by condition — ordered by effect size",
             fontsize=14, fontweight="bold", y=0.999)
fig.tight_layout(rect=[0, 0.025, 1, 1])
savefig(fig, "04_adt_by_condition", cfg)

In [ ]:
# =============================== FIGURE ====================================
# Condition effects across the whole panel, as a heatmap. Same information as
# the violin figure, compressed so the pattern across all 20 features is
# readable at once.
#
# A) Mean CLR by condition — absolute level, with the matched isotype mean
#    subtracted so features are comparable to each other despite differing
#    backgrounds.
# B) Change vs Control — the effect itself, diverging scale.
#
# Ordered by effect size, same as the violin figure, so the two can be read
# together.

adt_annot = panels["adt"]["annotations"]
order_f = struct.sort_values("cond_range_clr", ascending=False).index.tolist()

# background-subtracted level: each feature minus its own matched isotype mean
iso_mean = pd.Series({t: clr[iso_map[t]].mean() for t in order_f})
level = means[order_f].T.sub(iso_mean, axis=0)      # features x conditions
change = delta[order_f].T                            # features x conditions

# row labels carry the annotation
row_labels = [f"{t}  —  {adt_annot.get(t, '')}" for t in order_f]

fig, ax = plt.subplot_mosaic(
    """
    AB
    """,
    figsize=(15, 9),
    gridspec_kw={"width_ratios": [1, 1]},
)

# ---------------------------------- A --------------------------------------
sns.heatmap(level, ax=ax["A"], cmap="viridis", annot=True, fmt=".2f",
            annot_kws={"size": 7}, linewidths=0.4, linecolor="white",
            cbar_kws={"label": "mean CLR above matched isotype", "shrink": 0.6},
            yticklabels=row_labels, xticklabels=cond_order)
ax["A"].tick_params(axis="y", labelsize=7.5, rotation=0)
ax["A"].tick_params(axis="x", labelsize=9, rotation=25)

# ---------------------------------- B --------------------------------------
# Symmetric colour limits so zero sits at the centre of the diverging map —
# otherwise the colour of "no change" drifts with the data range.
vmax = np.abs(change.values).max()
sns.heatmap(change, ax=ax["B"], cmap="RdBu_r", center=0,
            vmin=-vmax, vmax=vmax, annot=True, fmt="+.2f",
            annot_kws={"size": 7}, linewidths=0.4, linecolor="white",
            cbar_kws={"label": "Δ CLR vs Control", "shrink": 0.6},
            yticklabels=False, xticklabels=cond_order)
ax["B"].tick_params(axis="x", labelsize=9, rotation=25)

titles = {
    "A": "surface level (isotype-subtracted)",
    "B": "change vs Control",
}
for label, a in ax.items():
    a.set_title(f"{label}) {titles.get(label, '')}", loc="left",
                fontweight="bold", fontsize=12, pad=10)

fig.suptitle("ADT panel: condition effects", fontsize=14, fontweight="bold",
             y=0.995)
fig.tight_layout()
savefig(fig, "04_adt_condition_heatmap", cfg)

In [ ]:
# ---- persist for nb05 ------------------------------------------------------
# The feature QC table and the ADT cluster labels are the two things
# downstream work needs. The CLR matrix itself is cheap to recompute from the
# .h5mu, so it isn't written.

# per-feature QC: background separation, detection rate, and structure in
# both level and detection — the evidence that no feature is dead
struct.to_csv(P.tables / "04_adt_feature_qc.csv")

# condition effects, so nb05 can ask whether perturbation effects run with or
# against the baseline condition response
delta.T.to_csv(P.tables / "04_adt_condition_delta.csv")

# ADT-derived cluster labels alongside the RNA labels, aligned by barcode
adt_labels = pd.DataFrame({
    "adt_cluster": adt_c.obs["adt_cluster"].astype(str),
    "rna_cluster": adt_c.obs["rna_cluster"].astype(str),
}, index=adt_c.obs_names)
adt_labels[["adt_umap1", "adt_umap2"]] = adt_c.obsm["X_umap"]
adt_labels.to_parquet(P.data_interim / "04_adt_clusters.parquet")

print(f"wrote:\n"
      f"  {P.tables / '04_adt_feature_qc.csv'}\n"
      f"  {P.tables / '04_adt_condition_delta.csv'}\n"
      f"  {P.data_interim / '04_adt_clusters.parquet'}")

import session_info
session_info.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp

# ---- per-cell geometric mean of raw ADT counts ----------------------------
# geometric mean of (x+1) across all 24 features per cell
# CLR divides each feature by this, so its distribution tells you how much
# the centering factor varies across cells
raw_adt = adt.layers["counts"] if "counts" in adt.layers else adt.X
raw_arr = np.asarray(
    raw_adt.todense() if sp.issparse(raw_adt) else raw_adt
)                                                    # cells x 24

# log of geometric mean = mean of logs
log_geom_mean = np.log(raw_arr + 1).mean(axis=1)    # one value per cell
print(f"per-cell log-geomean: mean={log_geom_mean.mean():.3f}, "
      f"sd={log_geom_mean.std():.3f}, "
      f"range=[{log_geom_mean.min():.3f}, {log_geom_mean.max():.3f}]")

# ---- CLR matrix for the feature distributions ----------------------------
clr_arr = np.asarray(
    adt_n[:, targets].X.todense()
    if sp.issparse(adt_n[:, targets].X)
    else adt_n[:, targets].X
)
clr_df = pd.DataFrame(clr_arr, index=adt_n.obs_names, columns=targets)

In [ ]:
# =============================== FIGURE ====================================
# ADT: per-cell centering factor and CLR distributions.
# A) spans full width — the centering factor check.
# B onwards: 4 features per row, 2 overlaid per panel.

from scipy.stats import gaussian_kde

# build the mosaic string dynamically
# row 0: AAAA (full width)
# remaining rows: 4 panels each, labelled B-U (up to 20 features / 2 = 10 panels)
feat_pairs = [(targets[i], targets[i+1] if i+1 < len(targets) else None)
              for i in range(0, len(targets), 2)]
n_pairs = len(feat_pairs)
n_rows  = int(np.ceil(n_pairs / 4))

# assign letters B onwards
letters = [chr(66 + i) for i in range(n_pairs)]

# build mosaic rows
mosaic_rows = ["AAAA"]
for row in range(n_rows):
    chunk = letters[row*4 : row*4 + 4]
    while len(chunk) < 4:
        chunk.append(".")          # pad with empty if not divisible by 4
    mosaic_rows.append("".join(chunk))

mosaic_str = "\n".join(mosaic_rows)
print("mosaic:\n", mosaic_str)

fig, ax = plt.subplot_mosaic(
    mosaic_str,
    figsize=(16, 3.5 * (1 + n_rows)),
)

# ---- A: per-cell centering factor ----------------------------------------
ax["A"].hist(log_geom_mean, bins=80, color="#4c9f70",
             edgecolor="none", alpha=0.85)
ax["A"].axvline(np.median(log_geom_mean), ls="--", c="k", lw=1.5,
                label=f"median = {np.median(log_geom_mean):.2f}")
ax["A"].set_xlabel("log geometric mean of raw ADT counts per cell")
ax["A"].set_ylabel("number of cells")
ax["A"].legend(fontsize=9)

# ---- B onwards: 2 features overlaid per panel ----------------------------
xs = np.linspace(clr_df.values.min(),
                 np.percentile(clr_df.values, 99.5), 300)
bw = 0.15
two_colors = ["#1F6FB2", "#C2185B"]

for letter, (f1, f2) in zip(letters, feat_pairs):
    a = ax[letter]
    for feat, c in zip([f for f in [f1, f2] if f is not None], two_colors):
        v = clr_df[feat].values
        kde = gaussian_kde(v, bw_method=bw)
        a.fill_between(xs, kde(xs), alpha=0.22, color=c)
        a.plot(xs, kde(xs), lw=2.0, color=c, label=feat)
        a.axvline(np.median(v), ls="--", c=c, lw=0.9, alpha=0.7)
    a.set_yticks([])
    a.set_xlabel("CLR")
    a.legend(fontsize=7.5, loc="upper right", framealpha=0.85)

# hide empty placeholder panels
for k, a in ax.items():
    if k == ".":
        a.set_axis_off()

# panel letters
all_labels = {"A": "per-cell centering factor"} | {l: "" for l in letters}
for label, a in ax.items():
    if label == ".":
        continue
    a.set_title(f"{label})", loc="left", fontweight="bold",
                fontsize=11, pad=6)

fig.suptitle("ADT: per-cell centering factor and CLR distributions\n"
             "(2 features overlaid per panel, dashed = median)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
savefig(fig, "04_adt_clr_check", cfg)